# 호텔 룸 3D Gaussian Splatting (Google Colab)

사진(50장 이상 권장) 또는 동영상(1~2분 walkthrough)으로 방을 3D 가우시안 스플래팅으로 재구성합니다.

**사용 전 준비**
1. 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. 구글 드라이브 최상위에 `gsplat_input` 폴더를 만들고 사진들(또는 `room.mp4` 동영상 1개)을 업로드
3. 아래 셀을 위에서부터 순서대로 실행 (▶ 버튼 또는 Shift+Enter)

**소요 시간(무료 T4 기준)**: 설치 ~10분, COLMAP ~10-30분(사진 수에 비례), 학습 ~20-40분

In [ ]:
# 1) GPU 확인 — "Tesla T4"가 보여야 합니다. 안 보이면 런타임 유형을 다시 확인하세요.
!nvidia-smi

In [ ]:
# 2) 설치 (약 10분) — nerfstudio(gsplat 백엔드) + COLMAP
!pip install -q nerfstudio
!apt-get install -y -q colmap ffmpeg
print("설치 완료")

In [ ]:
# 3) 구글 드라이브 연결 — 팝업에서 계정 선택 후 허용
from google.colab import drive
drive.mount('/content/drive')

import os
INPUT_DIR = '/content/drive/MyDrive/gsplat_input'
assert os.path.isdir(INPUT_DIR), 'gsplat_input 폴더가 드라이브에 없습니다. 만들고 사진/영상을 넣어주세요.'
print('입력 파일:', sorted(os.listdir(INPUT_DIR))[:10], '...')

## 4) 데이터 전처리 (COLMAP — 카메라 위치 자동 추정)

아래 **둘 중 하나만** 실행하세요.
- 4-A: 사진 여러 장을 올린 경우
- 4-B: 동영상(`room.mp4`)을 올린 경우 (프레임 자동 추출)

마지막 출력에 `Colmap matched X images`가 나옵니다. **X가 원본의 80% 미만이면** 사진 간 겹침이 부족한 것이니 재촬영을 권합니다.

In [ ]:
# 4-A) 사진 입력
!ns-process-data images --data {INPUT_DIR} --output-dir /content/processed

In [ ]:
# 4-B) 동영상 입력 (파일명이 다르면 room.mp4 부분 수정)
!ns-process-data video --data {INPUT_DIR}/room.mp4 --num-frames-target 150 --output-dir /content/processed

In [ ]:
# 5) 학습 (무료 T4 기준 20~40분. 처음 실행 시 gsplat CUDA 컴파일로 몇 분 더 걸립니다)
# 품질을 더 올리려면 30000, 빨리 보려면 7000으로 조정
!ns-train splatfacto \
  --data /content/processed \
  --max-num-iterations 15000 \
  --vis tensorboard \
  --output-dir /content/outputs

In [ ]:
# 6) 결과 내보내기 (.ply) 후 드라이브에 저장
import glob, shutil
config = sorted(glob.glob('/content/outputs/**/config.yml', recursive=True))[-1]
print('사용할 config:', config)
!ns-export gaussian-splat --load-config {config} --output-dir /content/export

out = '/content/drive/MyDrive/gsplat_result.ply'
shutil.copy(glob.glob('/content/export/*.ply')[0], out)
print('저장 완료:', out)

## 7) 결과 보기

드라이브의 `gsplat_result.ply`를 내려받아 **SuperSplat 편집기**(https://superspl.at/editor)에 드래그하면
브라우저에서 바로 3D로 둘러볼 수 있습니다. 잡티(floater) 제거·자르기·압축(.splat 변환)도 SuperSplat에서 가능합니다.

---
**문제 해결**
- COLMAP에서 매칭 실패/이미지 대부분 탈락 → 사진 겹침 부족. 동영상으로 재촬영 권장 (아래 촬영 팁 참고)
- 학습 중 세션 끊김 → 무료 Colab 세션 한도. `--max-num-iterations 7000`으로 낮춰 재시도
- 결과에 떠다니는 잡티 → SuperSplat에서 선택 삭제, 또는 iteration을 30000으로 늘려 재학습

**촬영 팁 (재촬영할 경우)**
- 폰 동영상으로 방을 **천천히** 한 바퀴 + 높이를 바꿔 한 바퀴 더 (총 1~2분)
- 인접 프레임 간 70% 이상 겹치게, 급회전 금지
- 조명 고정, 흔들림 최소화, 거울·유리·순백 벽면은 3DGS의 약점이니 과도한 기대 금지